In [3]:
%%writefile teste_nivelamento.py

import random
import math
import unittest
import pandas as pd

def gerar_datasets():
    """Gera duas listas de dicionários simulando tabelas de um banco de dados."""
    # A seed fixa ajuda na reprodutibilidade, mas os testes abaixo não dependem
    # de valores fixados no código (hardcoded), usando o Pandas para validação.
    random.seed(42)
    n = 200
    ids = list(range(1, n + 1))

    tabela_notas = []
    for i in ids:
        tabela_notas.append({
            'aluno_id': i,
            'horas_estudo': round(random.gauss(10, 3), 1),
            'nota_programacao': round(random.gauss(7, 1.5), 1),
            'turno': random.choice(['Matutino', 'Noturno'])
        })

    # Inserindo valores nulos (None) propositais
    for i in range(10, 20):
        tabela_notas[i]['nota_programacao'] = None

    ids_embaralhados = ids[:]
    random.shuffle(ids_embaralhados)
    tabela_presenca = []
    for i in ids_embaralhados:
        tabela_presenca.append({
            'aluno_id': i,
            'presenca': round(random.gauss(80, 10), 1)
        })

    return tabela_notas, tabela_presenca

# =======================================================
# ÁREA DO ALUNO - PREENCHA AS FUNÇÕES ABAIXO EM PYTHON PURO
# =======================================================

def questao_1_manipular_dados(tb_notas, tb_presenca):
    """
    Realize um INNER JOIN manual entre as duas listas de dicionários
    usando a chave 'aluno_id'.
    Filtre o resultado mantendo APENAS alunos com 'presenca' >= 75.0.
    Retorne uma nova lista de dicionários com os dados combinados.
    """

    resultado = []

    for registro_nota in tb_notas:
        for registro_presenca in tb_presenca:

            if registro_nota["aluno_id"] == registro_presenca["aluno_id"]:
                if registro_presenca["presenca"] >= 75.0:
                    novo_dado = registro_nota.copy()
                    novo_dado["presenca"] = registro_presenca["presenca"]
                    resultado.append(novo_dado)

    return resultado


def questao_2_estatistica(tb_notas):
    """
    Calcule a média e o desvio padrão amostral da chave 'horas_estudo'.
    Dica: o desvio padrão amostral divide pela quantidade (N - 1).
    Retorne a tupla: (media, desvio_padrao).
    """

    horas = []

    for registro in tb_notas:
        horas.append(registro["horas_estudo"])

    quantidade = len(horas)

    media = sum(horas) / quantidade

    soma = 0

    for valor in horas:
        soma += (valor - media) ** 2

    variancia = soma / (quantidade - 1)

    desvio = math.sqrt(variancia)

    return media, desvio


def questao_3_divisao_manual(dados, proporcao_teste=0.2):
    """
    Divida a lista de dados em treino e teste MANUALMENTE usando fatiamento (slicing).
    Separe os primeiros elementos para treino e o final para teste.
    Retorne a tupla: (lista_treino, lista_teste).
    """

    quantidade = len(dados)

    qtd_teste = int(quantidade * proporcao_teste)

    inicio_teste = quantidade - qtd_teste

    treino = dados[:inicio_teste]
    teste = dados[inicio_teste:]

    return treino, teste


def questao_4_dados_nulos(tb_notas):
    """
    Identifique os valores nulos (None) na chave 'nota_programacao'
    e substitua-os pela média das notas válidas desta mesma chave.
    Retorne a lista atualizada.
    """

    notas_validas = []

    for registro in tb_notas:
        nota = registro["nota_programacao"]

        if nota is not None:
            notas_validas.append(nota)

    media = sum(notas_validas) / len(notas_validas)

    resultado = []

    for registro in tb_notas:
        novo_registro = registro.copy()

        if novo_registro["nota_programacao"] is None:
            novo_registro["nota_programacao"] = media

        resultado.append(novo_registro)

    return resultado


def questao_5_calculo_erro(y_true, y_pred):
    """
    Recebe duas listas de mesmo tamanho: valores reais e predições.
    Calcule o Erro Quadrático Médio (MSE).
    Retorne o valor numérico do MSE.
    """

    total = 0

    for real, previsto in zip(y_true, y_pred):
        erro = real - previsto
        total += erro ** 2

    mse = total / len(y_true)

    return mse


def questao_6_agrupamento(tb_notas):
    """
    Agrupe os dados pela chave 'turno' e calcule a média de
    'horas_estudo' para cada turno.
    Retorne um dicionário no formato: {'Matutino': media, 'Noturno': media}.
    """

    dados_turnos = {}

    for registro in tb_notas:
        turno_atual = registro["turno"]

        if turno_atual not in dados_turnos:
            dados_turnos[turno_atual] = []

        dados_turnos[turno_atual].append(registro["horas_estudo"])

    resultado = {}

    for turno, horas in dados_turnos.items():
        resultado[turno] = sum(horas) / len(horas)

    return resultado

# =======================================================
# TESTES DE VALIDAÇÃO AUTOMÁTICA (NÃO ALTERE ESTA PARTE)
# =======================================================
class TestNivelamentoPythonPuro(unittest.TestCase):
    def setUp(self):
        # Gerando cópias novas para cada teste
        self.tb_notas, self.tb_presenca = gerar_datasets()
        # Convertendo para DataFrames estritamente para o cálculo do gabarito dinâmico
        self.df_notas = pd.DataFrame(self.tb_notas)
        self.df_presenca = pd.DataFrame(self.tb_presenca)

    def test_q1_manipulacao(self):
        try:
            # Gabarito Dinâmico
            df_merged = pd.merge(self.df_notas, self.df_presenca, on='aluno_id', how='inner')
            df_esperado = df_merged[df_merged['presenca'] >= 75.0]

            resultado = questao_1_manipular_dados(self.tb_notas, self.tb_presenca)

            self.assertIsNotNone(resultado)
            self.assertTrue(isinstance(resultado, list))
            self.assertEqual(len(resultado), len(df_esperado), "A quantidade de alunos após o join e filtro está incorreta.")
            self.assertIn('presenca', resultado[0], "A chave 'presenca' não foi encontrada após o Join.")
            self.assertTrue(all(d.get('presenca', 0) >= 75.0 for d in resultado), "Há alunos com presença inferior a 75.0.")
        except Exception as e: self.fail(f"Erro na Questão 1: {e}")

    def test_q2_estatistica(self):
        try:
            # Gabarito Dinâmico
            esperado_media = self.df_notas['horas_estudo'].mean()
            esperado_std = self.df_notas['horas_estudo'].std(ddof=1)

            media, std = questao_2_estatistica(self.tb_notas)

            self.assertAlmostEqual(media, esperado_media, places=2, msg="A média está incorreta.")
            self.assertAlmostEqual(std, esperado_std, places=2, msg="O desvio padrão está incorreto.")
        except Exception as e: self.fail(f"Erro na Questão 2: {e}")

    def test_q3_divisao(self):
        try:
            treino, teste = questao_3_divisao_manual(self.tb_notas, 0.2)

            tamanho_esperado_teste = int(len(self.tb_notas) * 0.2)
            self.assertEqual(len(treino) + len(teste), len(self.tb_notas), "A soma do treino e teste não bate com o total.")
            self.assertEqual(len(teste), tamanho_esperado_teste, "A proporção do conjunto de teste está incorreta.")
        except Exception as e: self.fail(f"Erro na Questão 3: {e}")

    def test_q4_nulos(self):
        try:
            # Gabarito Dinâmico (o Pandas ignora NaNs no cálculo da média automaticamente)
            media_esperada = self.df_notas['nota_programacao'].mean()

            resultado = questao_4_dados_nulos(self.tb_notas)
            nulos = [d for d in resultado if d['nota_programacao'] is None]

            self.assertEqual(len(nulos), 0, "Ainda existem valores nulos (None) na lista.")
            # O índice 15 era sabidamente nulo na geração dos dados
            self.assertAlmostEqual(resultado[15]['nota_programacao'], media_esperada, places=2,
                                   msg="A substituição dos valores nulos não utilizou a média correta.")
        except Exception as e: self.fail(f"Erro na Questão 4: {e}")

    def test_q5_calculo(self):
        try:
            # Gerando listas randômicas para testar
            y_true = [random.uniform(0, 10) for _ in range(15)]
            y_pred = [random.uniform(0, 10) for _ in range(15)]

            # Gabarito Dinâmico
            serie_true = pd.Series(y_true)
            serie_pred = pd.Series(y_pred)
            mse_esperado = ((serie_true - serie_pred) ** 2).mean()

            res = questao_5_calculo_erro(y_true, y_pred)
            self.assertAlmostEqual(res, mse_esperado, places=3, msg="O cálculo do MSE falhou.")
        except Exception as e: self.fail(f"Erro na Questão 5: {e}")

    def test_q6_agrupamento(self):
        try:
            # Gabarito Dinâmico
            agrupado_esperado = self.df_notas.groupby('turno')['horas_estudo'].mean().to_dict()

            res = questao_6_agrupamento(self.tb_notas)

            self.assertTrue(isinstance(res, dict), "O retorno deve ser um dicionário.")
            self.assertIn('Matutino', res)
            self.assertIn('Noturno', res)
            self.assertAlmostEqual(res['Matutino'], agrupado_esperado['Matutino'], places=2)
            self.assertAlmostEqual(res['Noturno'], agrupado_esperado['Noturno'], places=2)
        except Exception as e: self.fail(f"Erro na Questão 6: {e}")

if __name__ == '__main__':
    print("\n=== TESTE DE NIVELAMENTO - AVALIAÇÃO DINÂMICA ===")
    unittest.main(verbosity=2)

Overwriting teste_nivelamento.py


In [4]:
!python teste_nivelamento.py


=== TESTE DE NIVELAMENTO - AVALIAÇÃO DINÂMICA ===
test_q1_manipulacao (__main__.TestNivelamentoPythonPuro.test_q1_manipulacao) ... ok
test_q2_estatistica (__main__.TestNivelamentoPythonPuro.test_q2_estatistica) ... ok
test_q3_divisao (__main__.TestNivelamentoPythonPuro.test_q3_divisao) ... ok
test_q4_nulos (__main__.TestNivelamentoPythonPuro.test_q4_nulos) ... ok
test_q5_calculo (__main__.TestNivelamentoPythonPuro.test_q5_calculo) ... ok
test_q6_agrupamento (__main__.TestNivelamentoPythonPuro.test_q6_agrupamento) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.038s

OK


Questão 7 — [x**2 for x in range(5) if x % 2 != 0]

A expressão range(5) cria os valores de 0 até 4: 0, 1, 2, 3, 4.

A condição x % 2 != 0 verifica quais números são ímpares, então permanecem apenas 1 e 3.

Depois, cada número é elevado ao quadrado:

1² = 1
3² = 9

Portanto, o resultado será [1, 9].

Resposta: alternativa B.
--------------------------------------
Questão 8 — Qual operação mantém apenas os registros com a chave presente nas duas estruturas?

A operação que faz isso é o Inner Join.

Ele compara as duas estruturas e mantém somente os registros que possuem uma correspondência nas duas. Ou seja, se um registro existir apenas em uma das estruturas, ele não fará parte do resultado.

Resposta: alternativa B — Inner Join.